In [5]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from itertools import combinations
import pandas as pd
import gc
from tqdm import tqdm
import scipy.stats
import os

# Configuration
MODEL_ID = "allenai/OLMoE-1B-7B-0924"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
LAYER_TO_EVALUATE = 0
NUM_CALIBRATION_SAMPLES = 256
MAX_LENGTH = 512
OFFLOAD_DIR = "offload_weights"

if not os.path.exists(OFFLOAD_DIR):
    os.makedirs(OFFLOAD_DIR)

torch.manual_seed(SEED)

# Memory sweep
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    offload_folder=OFFLOAD_DIR,
    low_cpu_mem_usage=True
)
model.eval()

# Locate MoE Layers
moe_layers = [m for m in model.modules() if "OlmoeSparseMoEBlock" in str(type(m))]
num_experts = model.config.num_experts
print(f"Discovered {len(moe_layers)} MoE layers. Experts per layer: {num_experts}")

# Prepare Calibration Data
dataset = load_dataset("salesforce/wikitext", "wikitext-2-raw-v1", split="test")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Filter empty sequences
calib_data = [x for x in tokenized_dataset if x["input_ids"].shape[0] > 0 and x["input_ids"][0] != tokenizer.pad_token_id][:NUM_CALIBRATION_SAMPLES]
calib_loader = torch.utils.data.DataLoader(calib_data, batch_size=2)
print(f"Calibration Dataset ready: {len(calib_data)} samples.")

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [2]:
def get_expert_params(target_block, idx):
    """Extracts a copy of a single expert's weights from the concatenated tensor."""
    params = {}
    for name, param in target_block.experts.named_parameters():
        params[name] = param.data[idx].detach().clone()
    return params

def compute_metrics_and_oracle(model, calib_loader, target_block, idx_a, idx_b, baseline_logits_list):
    """Computes similarity and capability loss (Oracle) for merging A and B."""

    # 1. Compute Similarity Metrics (Cosine & L2)
    params_a_list, params_b_list = [], []
    backup_a = {}

    for name, param in target_block.experts.named_parameters():
        # Slice out a and b
        a_tensor = param.data[idx_a].detach()
        b_tensor = param.data[idx_b].detach()

        params_a_list.append(a_tensor.flatten())
        params_b_list.append(b_tensor.flatten())

        # Backup A for restoration later
        backup_a[name] = a_tensor.clone()

    vec_a = torch.cat(params_a_list)
    vec_b = torch.cat(params_b_list)

    w_dist = torch.norm(vec_a - vec_b).item()
    w_cos = F.cosine_similarity(vec_a.unsqueeze(0), vec_b.unsqueeze(0)).item()

    # 2. Execute Merge (Average A and B, write into A's slice)
    for name, param in target_block.experts.named_parameters():
        merged_tensor = (backup_a[name] + param.data[idx_b].detach()) / 2.0
        # CRITICAL FIX: Write specifically to the index slice, not the whole tensor
        param.data[idx_a].copy_(merged_tensor)

    # 3. Oracle Evaluation Loop
    kl_total = 0.0
    agree_total = 0.0
    num_batches = len(calib_loader)

    with torch.no_grad():
        for batch_idx, batch in enumerate(calib_loader):
            input_ids = batch["input_ids"].to(DEVICE)
            out = model(input_ids)
            orig_logits = baseline_logits_list[batch_idx].to(DEVICE)

            # CRITICAL FIX: Flatten Batch and Sequence dimensions for KL Divergence
            out_log_probs = F.log_softmax(out.logits, dim=-1).view(-1, out.logits.size(-1))
            orig_probs = F.softmax(orig_logits, dim=-1).view(-1, orig_logits.size(-1))

            kl = F.kl_div(out_log_probs, orig_probs, reduction='batchmean')
            kl_total += kl.item()

            # Top-1 Agreement
            agree_total += (out.logits.argmax(-1) == orig_logits.argmax(-1)).float().mean().item()

    # 4. Restore Model Architecture (Revert A to original state)
    for name, param in target_block.experts.named_parameters():
        param.data[idx_a].copy_(backup_a[name])

    return w_dist, w_cos, (kl_total / num_batches), (agree_total / num_batches)

In [3]:
target_block = moe_layers[LAYER_TO_EVALUATE]

# Generate Baseline Logits ONCE to avoid redundant compute
print("Generating baseline logits to establish Ground Truth...")
baseline_logits_list = []
with torch.no_grad():
    for batch in tqdm(calib_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        # Store on CPU to prevent VRAM overflow during the combination loop
        baseline_logits_list.append(model(input_ids).logits.cpu())

# Execute Pairwise Merge Evaluations
results = []
pairs = list(combinations(range(num_experts), 2))

print(f"Evaluating {len(pairs)} expert pairs for Layer {LAYER_TO_EVALUATE}...")
for a, b in tqdm(pairs):
    w_dist, w_cos, kl, agree = compute_metrics_and_oracle(
        model,
        calib_loader,
        target_block,
        a,
        b,
        baseline_logits_list
    )

    results.append({
        "Layer": LAYER_TO_EVALUATE,
        "Expert_A": a,
        "Expert_B": b,
        "Weight_Distance": w_dist,
        "Weight_Cosine": w_cos,
        "Oracle_KL": kl,
        "Top1_Agreement": agree
    })

    if a % 5 == 0:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

# Analysis Output
df = pd.DataFrame(results)
df.to_csv("olmoe_merge_metrics.csv", index=False)

print("\n--- RESULTS ANALYSIS ---")
pearson_r, _ = scipy.stats.pearsonr(df["Weight_Cosine"], df["Oracle_KL"])
spearman_r, _ = scipy.stats.spearmanr(df["Weight_Cosine"], df["Oracle_KL"])

print(f"Pearson Correlation (Cosine vs KL Divergence): {pearson_r:.4f}")
print(f"Spearman Rank Correlation (Cosine vs KL Divergence): {spearman_r:.4f}")
print("If correlation is highly negative, cosine similarity is a valid proxy for merge capability retention.")

NameError: name 'moe_layers' is not defined